# Did the online shopping boom after 2020 last?

What 5,014 real US Amazon accounts bought between 2018 and 2022.

This notebook produces every number behind the case study in the README. The data is *Open e-commerce 1.0* (Berke et al., 2024). You download it yourself (see the README). It is shared for research and must not be used to identify participants, so everything here is aggregated: nothing at the level of a single person leaves this notebook.

Run the cells from top to bottom. The tables are saved in `results/`, and `scripts/make_figures.py` draws the figures from them.

## Setup

`B` is how many times I re-draw the customers to see how much a number can move by chance (the bootstrap). `MIN_N` is the smallest customer group I am willing to show. The CPI values are the US Consumer Price Index (BLS, via FRED), used to take inflation out of every amount, so all money is in 2019 dollars.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

buy_path = "data/amazon-purchases.csv"
svy_path = "data/survey.csv"
out = Path("results")
out.mkdir(exist_ok=True)

B = 1000          # bootstrap draws
MIN_N = 300       # smallest group we show
rng = np.random.default_rng(42)
uid = "Survey ResponseID"
years = [2018, 2019, 2020, 2021, 2022]

# CPI-U all items, seasonally adjusted, Jan 2018 - Oct 2022 (BLS via FRED, CPIAUCSL)
cpi_vals = [
    248.859, 249.529, 249.577, 250.227, 250.792, 251.018, 251.214, 251.663, 252.182, 252.772, 252.594, 252.767,
    252.561, 253.319, 254.277, 255.233, 255.296, 255.213, 255.802, 256.036, 256.430, 257.155, 257.879, 258.630,
    259.127, 259.250, 258.076, 256.032, 255.802, 257.042, 258.352, 259.316, 259.997, 260.319, 260.911, 262.045,
    262.687, 263.579, 264.961, 266.614, 268.383, 270.654, 271.903, 272.676, 273.910, 276.550, 278.919, 280.845,
    282.543, 284.500, 287.674, 288.561, 291.298, 294.957, 294.913, 295.097, 296.349, 298.007,
]
cpi = pd.Series(cpi_vals, index=pd.period_range("2018-01", periods=len(cpi_vals), freq="M"))
base = cpi["2019"].mean()      # everything is in 2019 dollars

## Load and clean

Keep January 2018 to October 2022, work out spend (price × quantity), and put every amount in 2019 dollars. Gift cards are set aside: they are about 3% of spend and a few customers buy hundreds of them, which would distort the picture.

Then a small helper builds two numbers for each customer and year, using January to September only so that 2022 (which stops in October) is comparable: **spend**, and **shopping days** (days with at least one purchase).

In [ ]:
df = pd.read_csv(buy_path)
df["tgl"] = pd.to_datetime(df["Order Date"])
df = df[(df["tgl"] >= "2018-01-01") & (df["tgl"] <= "2022-10-31")].copy()
df["val"] = df["Purchase Price Per Unit"] * df["Quantity"]
df["ym"] = df["tgl"].dt.to_period("M")
df["real"] = df["val"] * base / df["ym"].map(cpi)
df["yr"] = df["tgl"].dt.year
df["mo"] = df["tgl"].dt.month
gc_pat = "gift card|gift code|digital code|amazon reload"
df["gc"] = df["Title"].fillna("").str.lower().str.contains(gc_pat)
dm = df[~df["gc"]].copy()

print("rows in period:", len(df), "| without gift cards:", len(dm))
print("users without gift cards:", dm[uid].nunique())

def arrays(d, ids=None):
    # spend (S) and shopping days (D) per customer per year, Jan-Sep only
    if ids is None:
        ids = np.sort(d[uid].unique())
    pos = pd.Series(np.arange(len(ids)), index=ids)
    d = d[(d["mo"] <= 9) & d[uid].isin(ids)]
    S = np.zeros((len(ids), 5))
    D = np.zeros((len(ids), 5))
    for k, y in enumerate(years):
        dy = d[d["yr"] == y]
        s = dy.groupby(uid)["real"].sum()
        S[pos[s.index].values, k] = s.values
        n_ = dy.groupby(uid)["tgl"].nunique()
        D[pos[n_.index].values, k] = n_.values
    return ids, S, D

ids, S, D = arrays(dm)
n = len(ids)
W = rng.multinomial(n, np.ones(n) / n, size=B).astype(float)   # bootstrap weights, B x n

def band(x, lo=2.5, hi=97.5):
    return np.percentile(x, [lo, hi], axis=0)

## 1. How much more does the average customer spend, and how much of it was already coming?

I compare each year with 2019, and with where the 2018 to 2019 growth would have led if it had just continued. Read the two together as a range: 2019 is the floor, the old trend is the ceiling. The second part builds the quarterly view used in Figure 1.

In [ ]:
m = S.mean(0)
bm = W @ S / n
g = m[1] / m[0] - 1                     # growth 2018 -> 2019 (before COVID)
cf = m[1] * (1 + g) ** np.arange(-1, 4)
gb = bm[:, 1] / bm[:, 0] - 1
cfb = bm[:, [1]] * (1 + gb[:, None]) ** np.arange(-1, 4)

rows = []
for k in (2, 3, 4):
    rows.append({
        "year": years[k],
        "real_spend_per_customer": m[k],
        "vs_2019_pct": (m[k] / m[1] - 1) * 100,
        "vs_2019_lo": band(bm[:, k] / bm[:, 1] - 1)[0] * 100,
        "vs_2019_hi": band(bm[:, k] / bm[:, 1] - 1)[1] * 100,
        "vs_old_trend_pct": (m[k] / cf[k] - 1) * 100,
        "vs_old_trend_lo": band(bm[:, k] / cfb[:, k] - 1)[0] * 100,
        "vs_old_trend_hi": band(bm[:, k] / cfb[:, k] - 1)[1] * 100,
    })
head = pd.DataFrame(rows).round(1)
head.to_csv(out / "01_headline.csv", index=False)
print("\n1. HEADLINE (Jan-Sep, 2019 dollars, per customer)")
print("growth 2018->2019:", round(g * 100, 1), "%")
print(head.to_string(index=False))

# quarterly view
dq = dm[dm["tgl"] < "2022-10-01"].copy()
dq["q"] = dq["tgl"].dt.to_period("Q")
tot = dq.groupby("q")["real"].sum()
act = dq.groupby("q")[uid].nunique()
percust = dq.groupby(["q", uid])["real"].sum().groupby("q").median()
qt = pd.DataFrame({
    "real_spend_per_customer": tot / n,
    "active_customers": act,
    "active_share_pct": act / n * 100,
    "real_spend_per_active": tot / act,
    "median_spend_per_active": percust,
})
qt["yoy_pct"] = qt["real_spend_per_customer"].pct_change(4) * 100
qt.round(1).to_csv(out / "01b_quarterly.csv")

## 2. Did the 2020 step-up last?

A customer's "normal" level is the average of 2018 and 2019. For customers who spent more than normal in 2020, I measure how much of the extra was still there in 2022. Then I follow both groups (spent more, did not) over the years, each against its own normal level.

In [ ]:
def retention(Sx, t, p, top=False, wts=None, boot=True):
    b = Sx[:, :2].mean(1)            # normal level = average of 2018 and 2019
    inc = Sx[:, t] - b
    mk = inc > (np.percentile(inc, 67) if top else 0)
    gain = mk * inc
    kept = mk * (Sx[:, p] - b)
    w1 = np.ones(len(inc)) if wts is None else wts
    est = (w1 @ kept) / (w1 @ gain)
    bs = (W @ kept) / (W @ gain) if boot and len(inc) == n else None
    return est, bs, mk.mean()

rows = []
for top in (False, True):
    est, bs, share = retention(S, 2, 4, top)
    lo, hi = band(bs)
    rows.append({"who": "top third of increases" if top else "everyone who increased in 2020",
                 "customers_pct": share * 100, "uplift_still_there_pct": est * 100,
                 "lo": lo * 100, "hi": hi * 100})
ret = pd.DataFrame(rows).round(1)
ret.to_csv(out / "02_persistence.csv", index=False)
print("\n2. HOW MUCH OF THE 2020 STEP-UP (vs 2018-19 average) IS STILL THERE IN 2022")
print(ret.to_string(index=False))

# customers who spent more in 2020 vs those who did not, indexed to their 2018-19 average = 100
nl = S[:, :2].mean(1)
up = S[:, 2] > nl
rows = []
for name, mk in (("Spent more in 2020", up), ("Did not spend more in 2020", ~up)):
    mean_g = (S * mk[:, None]).sum(0) / mk.sum()
    bg = (W @ (S * mk[:, None])) / (W @ mk.astype(float))[:, None]
    lvl = nl[mk].mean()
    idx = mean_g / lvl * 100
    bidx = ((W @ (S * mk[:, None])) / (W @ mk.astype(float))[:, None]) / ((W @ (nl * mk)) / (W @ mk.astype(float)))[:, None] * 100
    lo_, hi_ = band(bidx)
    for k, y in enumerate(years):
        rows.append({"group": name, "customers_pct": mk.mean() * 100, "year": y,
                     "real_spend_per_customer": mean_g[k], "index_2018_19": idx[k],
                     "lo": lo_[k], "hi": hi_[k]})
traj = pd.DataFrame(rows).round(1)
traj.to_csv(out / "02b_trajectories.csv", index=False)
print("\n2b. SPEND INDEXED TO 2018-19 AVERAGE = 100, BY GROUP")
print(traj[["group", "year", "index_2018_19", "lo", "hi"]].to_string(index=False))

## 3. Where did the growth come from?

Spend per customer = share of customers who shopped at all × shopping days per active customer × spend per shopping day. Taking logs turns the multiplication into an addition, so the three contributions can be compared. The values here are in log points; the README converts them into plain percentages.

In [ ]:
def comps(w, k):
    w = np.atleast_2d(w)
    act_ = (S[:, k] > 0).astype(float)
    a = (w @ act_) / w.sum(1)
    d = (w @ D[:, k]) / (w @ act_)
    s = (w @ S[:, k]) / (w @ D[:, k])
    return a, d, s

rows = []
for k in (2, 4):
    a0, d0, s0 = comps(np.ones(n), 1)
    a1, d1, s1 = comps(np.ones(n), k)
    pt = np.array([np.log(a1 / a0), np.log(d1 / d0), np.log(s1 / s0)]).ravel()
    ba0, bd0, bs0 = comps(W, 1)
    ba1, bd1, bs1 = comps(W, k)
    bt = np.column_stack([np.log(ba1 / ba0), np.log(bd1 / bd0), np.log(bs1 / bs0)])
    lo, hi = band(bt)
    for j, lab in enumerate(["Share of customers active", "Shopping days per active customer", "Spend per shopping day"]):
        rows.append({"vs 2019": years[k], "driver": lab, "log_change_pct": pt[j] * 100,
                     "lo": lo[j] * 100, "hi": hi[j] * 100,
                     "share_of_total_growth_pct": pt[j] / pt.sum() * 100})
dec = pd.DataFrame(rows).round(1)
dec.to_csv(out / "03_growth_drivers.csv", index=False)
print("\n3. GROWTH DRIVERS (log change vs 2019, Jan-Sep)")
print(dec.to_string(index=False))

## 4. Which categories gained, lost or kept their share?

Only categories bought by at least 100 customers in every year. Each label comes from bootstrap bands, using a stricter 99% band because 213 categories are tested at once. A smaller share does not mean less spend: everything was growing, so the last column compares each category's spend growth with the average.

In [ ]:
dk = dm[dm["Category"].notna() & (dm["mo"] <= 9)]
buyers = dk.groupby(["Category", "yr"])[uid].nunique().unstack(fill_value=0)
cats = list(buyers.index[(buyers >= 100).all(axis=1)])
print("\n4. categories with >=100 customers in every year:", len(cats))
pos = pd.Series(np.arange(n), index=ids)
cpos = {c: i for i, c in enumerate(cats)}
a = dk[dk["Category"].isin(cats)].groupby([uid, "Category", "yr"])["real"].sum().reset_index()
X = np.zeros((n, len(cats), 5))
X[a[uid].map(pos).values, a["Category"].map(cpos).values, a["yr"].values - 2018] = a["real"].values
T = np.zeros((n, 5))
t_ = dk.groupby([uid, "yr"])["real"].sum().reset_index()
T[t_[uid].map(pos).values, t_["yr"].values - 2018] = t_["real"].values
cover = X.sum((0, 1)) / T.sum(0) * 100
print("share of categorized spend covered, by year:", np.round(cover, 1))
pd.DataFrame({"year": years, "pct_of_categorized_spend_covered": cover}).round(1).to_csv(out / "04b_coverage.csv", index=False)

sh = X.sum(0) / T.sum(0) * 100                    # categories x years
num = (W @ X.reshape(n, -1)).reshape(B, len(cats), 5)
bsh = num / (W @ T)[:, None, :] * 100

def diff(s, i, j):
    return s[..., i] - s[..., j]

pairs = {"pre": (1, 0), "d20": (2, 1), "d22": (4, 1), "fade": (4, 2)}
est = {k: diff(sh, *v) for k, v in pairs.items()}
lo = {k: np.percentile(diff(bsh, *v), 0.5, axis=0) for k, v in pairs.items()}    # 99% band
hi = {k: np.percentile(diff(bsh, *v), 99.5, axis=0) for k, v in pairs.items()}
sig = lambda k, i: (lo[k][i] > 0) or (hi[k][i] < 0)

labels = []
for i in range(len(cats)):
    if sig("pre", i) and sig("d22", i) and np.sign(est["pre"][i]) == np.sign(est["d22"][i]):
        labels.append("Old trend, continued")
    elif est["d20"][i] > 0 and sig("d20", i):
        if lo["d22"][i] <= 0:
            labels.append("Temporary spike")
        elif hi["fade"][i] < 0:
            labels.append("Partly retained")
        else:
            labels.append("Sticky")
    elif est["d20"][i] < 0 and sig("d20", i):
        labels.append("Lost share, stayed lower" if hi["d22"][i] < 0 else "Lost share, then recovered")
    else:
        labels.append("No clear change")

names = {"ABIS_BOOK": "Books", "ABIS_MUSIC": "Music (physical)"}
avg_growth = (m[4] / m[1] - 1) * 100
cat = pd.DataFrame({
    "category": cats,
    "name": [names.get(c, c.replace("_", " ").title()) for c in cats],
    **{f"share_{y}": sh[:, k] for k, y in enumerate(years)},
    "chg_pre_2019": est["pre"], "chg_2020": est["d20"], "chg_2022": est["d22"],
    "chg_2020_lo": lo["d20"], "chg_2020_hi": hi["d20"],
    "chg_2022_lo": lo["d22"], "chg_2022_hi": hi["d22"],
    "spend_per_customer_2022_vs_2019_pct": (X[:, :, 4].sum(0) / X[:, :, 1].sum(0) - 1) * 100,
    "label": labels,
})
cat["vs_average_growth_pts"] = cat["spend_per_customer_2022_vs_2019_pct"] - avg_growth
cat = cat.round(2)
cat.sort_values("chg_2020", ascending=False).to_csv(out / "04_categories.csv", index=False)
print(cat["label"].value_counts().to_string())

## 5. Customer groups by recency, frequency and spend (RFM)

Each customer is scored on how recently, how often and how much they bought in the 12 months to 30 September 2022, then placed in a simple group. The last lines show how concentrated spend is.

In [ ]:
as_of = pd.Timestamp("2022-09-30")
w12 = dm[(dm["tgl"] > "2021-09-30") & (dm["tgl"] <= as_of)]
r = w12.groupby(uid).agg(last=("tgl", "max"), freq=("tgl", "nunique"), spend=("real", "sum"))
r["rec"] = (as_of - r["last"]).dt.days

def score(s, high_good):
    rk = s.rank(method="first", ascending=high_good)
    return pd.qcut(rk, 5, labels=[1, 2, 3, 4, 5]).astype(int)

r["R"] = score(r["rec"], False)
r["F"] = score(r["freq"], True)
r["M"] = score(r["spend"], True)
R, F = r["R"], r["F"]
r["segment"] = np.select(
    [(R >= 4) & (F >= 4), (R >= 3) & (F >= 3), (R >= 4) & (F <= 2), (R <= 2) & (F >= 4), (R <= 2) & (F <= 2)],
    ["Champions", "Loyal", "Recent, low frequency", "At risk", "Hibernating"],
    default="Need attention")
seg = r.groupby("segment").agg(customers=("spend", "size"), spend=("spend", "sum"),
                               median_spend=("spend", "median"), median_shopping_days=("freq", "median"),
                               avg_recency_days=("rec", "mean"))
seg["pct_of_customers"] = seg["customers"] / n * 100
seg["pct_of_spend"] = seg["spend"] / seg["spend"].sum() * 100
seg = seg.sort_values("pct_of_spend", ascending=False).round(1)
seg.to_csv(out / "05_rfm.csv")
none = n - len(r)
allsp = np.sort(np.concatenate([r["spend"].values, np.zeros(none)]))[::-1]
top10 = allsp[: int(n * 0.1)].sum() / allsp.sum() * 100
top20 = allsp[: int(n * 0.2)].sum() / allsp.sum() * 100
print("\n5. RFM, last 12 months to 30 Sep 2022 | customers with no purchase:", none)
print(seg.to_string())
print(f"top 10% of customers = {top10:.1f}% of spend | top 20% = {top20:.1f}% | "
      f"median {r['spend'].median():.0f} vs mean {r['spend'].mean():.0f}")

## 6. Do different kinds of customers behave differently?

The same two measures for groups by age, income, education, household size, gender and region. Groups with fewer than `MIN_N` customers are not shown. Age is as of the survey (late 2022), so part of the youngest group's growth is life stage, not COVID.

In [ ]:
sv = pd.read_csv(svy_path).set_index(uid).reindex(ids)
st = [("Alabama", "AL", "South"), ("Alaska", "AK", "West"), ("Arizona", "AZ", "West"), ("Arkansas", "AR", "South"),
      ("California", "CA", "West"), ("Colorado", "CO", "West"), ("Connecticut", "CT", "Northeast"),
      ("Delaware", "DE", "South"), ("District of Columbia", "DC", "South"), ("Florida", "FL", "South"),
      ("Georgia", "GA", "South"), ("Hawaii", "HI", "West"), ("Idaho", "ID", "West"), ("Illinois", "IL", "Midwest"),
      ("Indiana", "IN", "Midwest"), ("Iowa", "IA", "Midwest"), ("Kansas", "KS", "Midwest"),
      ("Kentucky", "KY", "South"), ("Louisiana", "LA", "South"), ("Maine", "ME", "Northeast"),
      ("Maryland", "MD", "South"), ("Massachusetts", "MA", "Northeast"), ("Michigan", "MI", "Midwest"),
      ("Minnesota", "MN", "Midwest"), ("Mississippi", "MS", "South"), ("Missouri", "MO", "Midwest"),
      ("Montana", "MT", "West"), ("Nebraska", "NE", "Midwest"), ("Nevada", "NV", "West"),
      ("New Hampshire", "NH", "Northeast"), ("New Jersey", "NJ", "Northeast"), ("New Mexico", "NM", "West"),
      ("New York", "NY", "Northeast"), ("North Carolina", "NC", "South"), ("North Dakota", "ND", "Midwest"),
      ("Ohio", "OH", "Midwest"), ("Oklahoma", "OK", "South"), ("Oregon", "OR", "West"),
      ("Pennsylvania", "PA", "Northeast"), ("Rhode Island", "RI", "Northeast"), ("South Carolina", "SC", "South"),
      ("South Dakota", "SD", "Midwest"), ("Tennessee", "TN", "South"), ("Texas", "TX", "South"),
      ("Utah", "UT", "West"), ("Vermont", "VT", "Northeast"), ("Virginia", "VA", "South"),
      ("Washington", "WA", "West"), ("West Virginia", "WV", "South"), ("Wisconsin", "WI", "Midwest"),
      ("Wyoming", "WY", "West")]
reg = {}
for nm, ab, rg in st:
    reg[nm.lower()] = rg
    reg[ab.lower()] = rg
sv["region"] = sv["Q-demos-state"].astype(str).str.strip().str.lower().map(reg)
print("\n6. customers with unknown region:", int(sv["region"].isna().sum()))

age_map = {"18 - 24 years": "18-24", "25 - 34 years": "25-34", "35 - 44 years": "35-44",
           "45 - 54 years": "45-54", "55 - 64 years": "55+", "65 and older": "55+"}
inc_map = {"Less than $25,000": "Under 50k USD", "$25,000 - $49,999": "Under 50k USD",
           "$50,000 - $74,999": "50k-99k USD", "$75,000 - $99,999": "50k-99k USD",
           "$100,000 - $149,999": "100k+ USD", "$150,000 or more": "100k+ USD"}
edu_map = {"Some high school or less": "High school or less", "High school diploma or GED": "High school or less",
           "Bachelor's degree": "Bachelor's", "Graduate or professional degree (MA, MS, MBA, PhD, JD, MD, DDS, etc)": "Graduate"}
sv["age"] = sv["Q-demos-age"].map(age_map)
sv["income"] = sv["Q-demos-income"].map(inc_map)
sv["education"] = sv["Q-demos-education"].map(edu_map)
sv["household"] = sv["Q-amazon-use-hh-size"].astype(str).map(
    {"1 (just me!)": "1 person", "2": "2 people", "3": "3 people", "4+": "4+ people"})
sv["gender"] = sv["Q-demos-gender"].where(sv["Q-demos-gender"].isin(["Female", "Male"]))

rows = []
for dim in ["age", "income", "education", "household", "gender", "region"]:
    for lev, ix in sv.groupby(dim).indices.items():
        if len(ix) < MIN_N:
            continue
        ng = len(ix)
        Wg = rng.multinomial(ng, np.ones(ng) / ng, size=B).astype(float)
        Sg = S[ix]
        ch = Sg[:, 4].sum() / Sg[:, 1].sum() - 1
        bch = (Wg @ Sg[:, 4]) / (Wg @ Sg[:, 1]) - 1
        inc = Sg[:, 2] - Sg[:, 1]
        mk = inc > 0
        rt = (mk * (Sg[:, 4] - Sg[:, 1])).sum() / (mk * inc).sum()
        brt = (Wg @ (mk * (Sg[:, 4] - Sg[:, 1]))) / (Wg @ (mk * inc))
        rows.append({"dimension": dim, "group": lev, "customers": ng,
                     "spend_2022_vs_2019_pct": ch * 100, "lo": band(bch)[0] * 100, "hi": band(bch)[1] * 100,
                     "uplift_kept_pct": rt * 100, "kept_lo": band(brt)[0] * 100, "kept_hi": band(brt)[1] * 100})
grp = pd.DataFrame(rows).round(1)
grp.to_csv(out / "06_groups.csv", index=False)
print(grp.to_string(index=False))

## 7. Do the main numbers survive other choices?

Gift cards kept in, the top 0.1% of line items removed, large quantities removed, and the sample re-weighted to match US adults by age and sex. If the answer barely moves, I trust it more.

In [ ]:
def rake(sv_):
    age3 = sv_["Q-demos-age"].map({"18 - 24 years": "18-34", "25 - 34 years": "18-34", "35 - 44 years": "35-54",
                                    "45 - 54 years": "35-54", "55 - 64 years": "55+", "65 and older": "55+"})
    sex = sv_["gender"]
    ok = (age3.notna() & sex.notna()).values
    w = np.where(ok, 1.0, 0.0)
    # population shares from the dataset paper (US census, adults 18+)
    for _ in range(100):
        for col, tg in ((age3, {"18-34": .294, "35-54": .323, "55+": .383}), (sex, {"Female": .51, "Male": .49})):
            for k, share in tg.items():
                mk = (col == k).values & ok
                if w[mk].sum() > 0:
                    w[mk] *= share * w.sum() / w[mk].sum()
    return w

def summary(Sx, Dx, wts=None):
    w1 = np.ones(len(Sx)) if wts is None else wts
    mm = (w1 @ Sx) / w1.sum()
    gg = mm[1] / mm[0] - 1
    est, _, _ = retention(Sx, 2, 4, wts=w1, boot=False)
    a19 = (w1 @ (Sx[:, 1] > 0)) / w1.sum()
    a22 = (w1 @ (Sx[:, 4] > 0)) / w1.sum()
    return {"spend_2022_vs_2019_pct": (mm[4] / mm[1] - 1) * 100,
            "spend_2022_vs_old_trend_pct": (mm[4] / (mm[1] * (1 + gg) ** 3) - 1) * 100,
            "uplift_kept_covid_pct": est * 100,
            "active_share_change_pts": (a22 - a19) * 100}

wr = rake(sv)
print("\n7. reweighting: effective sample size", round(wr.sum() ** 2 / (wr ** 2).sum()), "of", int((wr > 0).sum()))
ids_g, S_g, D_g = arrays(df)
trim = dm[dm["val"] <= dm["val"].quantile(0.999)]
_, S_t, D_t = arrays(trim, ids)
_, S_q, D_q = arrays(dm[dm["Quantity"] <= 10], ids)
rob = pd.DataFrame({
    "main": summary(S, D),
    "gift cards included": summary(S_g, D_g),
    "top 0.1% line items removed": summary(S_t, D_t),
    "quantity > 10 removed": summary(S_q, D_q),
    "reweighted to US age and sex": summary(S, D, wr),
}).T.round(1)
rob.to_csv(out / "07_robustness.csv")
print(rob.to_string())
print("\ndone. files are in:", out.resolve())

## Next

Run `python scripts/make_figures.py` from the project folder to draw the figures from the tables in `results/`.